## Challenge 1 
### Convert hex to base64

The string:

49276d206b696c6c696e6720796f757220627261696e206c696b65206120706f69736f6e6f7573206d757368726f6f6d

Should produce:



SSdtIGtpbGxpbmcgeW91ciBicmFpbiBsaWtlIGEgcG9pc29ub3VzIG11c2hyb29t

So go ahead and make that happen. You'll need to use this code for the rest of the exercises.

### Cryptopals Rule
Always operate on raw bytes, never on encoded strings. Only use hex and base64 for pretty-printing.

In [ ]:
import base64

In [ ]:
h = "49276d206b696c6c696e6720796f757220627261696e206c696b65206120706f69736f6e6f7573206d757368726f6f6d"
pt = bytes.fromhex(h)
enc = base64.b64encode(pt)
print(enc.decode())

## Challenge 2
### Fixed XOR


Write a function that takes two equal-length buffers and produces their XOR combination.

If your function works properly, then when you feed it the string:

1c0111001f010100061a024b53535009181c

... after hex decoding, and when XOR'd against:

686974207468652062756c6c277320657965

... should produce:

746865206b696420646f6e277420706c6179

In [ ]:
h1 = "1c0111001f010100061a024b53535009181c"
h2 = "686974207468652062756c6c277320657965"

d1 = bytes.fromhex(h1)
d2 = bytes.fromhex(h2)
assert len(d1) == len(d2), "fixed xor should use 2 buffer that have same length!"

h = ""
for b1, b2 in zip(d1, d2):
    h += hex(b1 ^ b2)[2:]
print(h)

## Challenge 3
### Single-byte XOR cipher

The hex encoded string:

1b37373331363f78151b7f2b783431333d78397828372d363c78373e783a393b3736

... has been XOR'd against a single character. Find the key, decrypt the message.

You can do this by hand. But don't: write code to do it for you.

How? Devise some method for "scoring" a piece of English plaintext. Character frequency is a good metric. Evaluate each output and choose the one with the best score.

### Achievement Unlocked
You now have our permission to make "ETAOIN SHRDLU" jokes on Twitter.

In [ ]:
def single_byte_xor(data: bytes, key: int):
    return bytes(bytearray([d ^ key for d in data]))

def score_english(text: bytes):
    ascii = [i for i in range(32,127)]
    letters = [69, 84, 65, 79, 73, 78, 83, 72, 82, 68, 76, 67, 85, 101, 116, 97, 111, 105, 110, 115, 104, 114, 100, 108, 99, 117, 32]
    penalty = [mins for mins in b"!~@#{}$%^&'`*"]
    score = 0

    for t in text:
        if t not in ascii:
            return 0
        if t in letters:
            match t:
                case 69 | 97:
                    score += 5
                case 84 | 111:
                    score += 3
                case 65 | 105:
                    score += 2
                case 32 | 110:
                    score += 4
            score += 1
        if t in penalty:
            score -= 2
    return score/len(text)

def crack_single_byte_xor(ciphertext: bytes, req_space=False):
    cand = []

    for k in range(256):
        p = b""
        p += single_byte_xor(ciphertext, k)

        score = score_english(p)
        if k == 0:
            cand = [ciphertext.hex(), k, p, score]
            continue
        if score < cand[3]:
            continue
        if req_space == False and (b" " not in p or b"  " in p):
            continue
        cand = [ciphertext.hex(), k, p, score]
    
    return tuple(cand)

In [ ]:
h = "1b37373331363f78151b7f2b783431333d78397828372d363c78373e783a393b3736"
ct = bytes.fromhex(h)
pt = crack_single_byte_xor(ct)
print(pt[2].decode())

## Challenge 4
### Detect single-character XOR


One of the 60-character strings in this file has been encrypted by single-character XOR.

Find it.

(Your code from #3 should help.)

In [ ]:
def single_byte_xor(data: bytes, key: int):
    return bytes(bytearray([d ^ key for d in data]))

def score_english(text: bytes):
    ascii = [i for i in range(32,127)] + [10,9,13]
    letters = [69, 84, 65, 79, 73, 78, 83, 72, 82, 68, 76, 67, 85, 101, 116, 97, 111, 105, 110, 115, 104, 114, 100, 108, 99, 117, 32, 10, 9, 13]
    penalty = [mins for mins in b"!~@#{}$%^&'`*"]
    score = 0

    for t in text:
        if t not in ascii:
            return 0
        if t in letters:
            match t:
                case 69 | 97:
                    score += 5
                case 84 | 111:
                    score += 3
                case 65 | 105:
                    score += 2
                case 32 | 110:
                    score += 3
            score += 1
        if t in penalty:
            score -= 3
    return score/len(text)

def crack_single_byte_xor(ciphertext: bytes, req_space=False):
    cand = []

    for k in range(256):
        p = b""
        p = single_byte_xor(ciphertext, k)

        score = score_english(p)
        if len(cand) == 0:
            cand = [ciphertext.hex(), k, p, score]
            continue
        if score < cand[3]:
            continue
        if req_space == False and (b" " not in p or b"  " in p):
            continue
        cand = [ciphertext.hex(), k, p, score]
    
    return tuple(cand)

In [ ]:
with open("/mnt/d/my-kisah/crypto/1.cryptopals/files/4.txt") as f:
    buff = f.readlines()

best = None
for h in buff:
    ct = bytes.fromhex(h.strip())

    cand = crack_single_byte_xor(ct)
    
    if best == None:
        best = cand
        continue
    if cand[3] > best[3]:
        best = cand

ct, key, pt, score = best
print(f"ciphertext  : {ct}")
print(f"key         : {key}")
print(f"plaintext   : {pt.decode().strip()}")
print(f"score       : {score}")

## Challenge 5
### Implement repeating-key XOR

Here is the opening stanza of an important work of the English language:


Burning 'em, if you ain't quick and nimble
I go crazy when I hear a cymbal

Encrypt it, under the key "ICE", using repeating-key XOR.

In repeating-key XOR, you'll sequentially apply each byte of the key; the first byte of plaintext will be XOR'd against I, the next C, the next E, then I again for the 4th byte, and so on.

It should come out to:

0b3637272a2b2e63622c2e69692a23693a2a3c6324202d623d63343c2a26226324272765272
a282b2f20430a652e2c652a3124333a653e2b2027630c692b20283165286326302e27282f

Encrypt a bunch of stuff using your repeating-key XOR function. Encrypt your mail. Encrypt your password file. Your .sig file. Get a feel for it. I promise, we aren't wasting your time with this.

In [ ]:
def repeating_key_xor(data: bytes, key: bytes) -> bytes:
    return bytes(bytearray([key[i%len(key)] ^ d for i, d in enumerate(data)]))

In [ ]:
pt = b"""Burning 'em, if you ain't quick and nimble
I go crazy when I hear a cymbal"""
key = b"ICE"
ct = repeating_key_xor(pt, key)
print(ct.hex())

## Challenge 6
### Break repeating-key XOR


#### It is officially on, now.
This challenge isn't conceptually hard, but it involves actual error-prone coding. The other challenges in this set are there to bring you up to speed. This one is there to qualify you. If you can do this one, you're probably just fine up to Set 6.

There's a file here. It's been base64'd after being encrypted with repeating-key XOR.

Decrypt it.

Here's how:

1. Let KEYSIZE be the guessed length of the key; try values from 2 to (say) 40.
2. Write a function to compute the edit distance/Hamming distance between two strings. The Hamming distance is just the number of differing bits. The distance between:

this is a test

and

wokka wokka!!!

3. For each KEYSIZE, take the first KEYSIZE worth of bytes, and the second KEYSIZE worth of bytes, and find the edit distance between them. Normalize this result by dividing by KEYSIZE.
4. The KEYSIZE with the smallest normalized edit distance is probably the key. You could proceed perhaps with the smallest 2-3 KEYSIZE values. Or take 4 KEYSIZE blocks instead of 2 and average the distances.
5. Now that you probably know the KEYSIZE: break the ciphertext into blocks of KEYSIZE length.
6. Now transpose the blocks: make a block that is the first byte of every block, and a block that is the second byte of every block, and so on.
7. Solve each block as if it was single-character XOR. You already have code to do this.
9. For each block, the single-byte XOR key that produces the best looking histogram is the repeating-key XOR key byte for that block. Put them together and you have the key.

This code is going to turn out to be surprisingly useful later on. Breaking repeating-key XOR ("Vigenere") statistically is obviously an academic exercise, a "Crypto 101" thing. But more people "know how" to break it than can actually break it, and a similar technique breaks something much more important.

#### No, that's not a mistake.
We get more tech support questions for this challenge than any of the other ones. We promise, there aren't any blatant errors in this text. In particular: the "wokka wokka!!!" edit distance really is 37.

In [ ]:
import base64

In [ ]:
def fixed_xor(b1: bytes, b2: bytes):
    if len(b1) - len(b2) != 0:
        raise Exception("buffer must have same length")
    return bytes(bytearray([x ^ y for x, y in zip(b1, b2)]))

def hamming_distance(b1: bytes, b2: bytes):
    xord = fixed_xor(b1, b2)
    return int.from_bytes(xord,"big").bit_count()

In [ ]:
def repeating_key_xor(data: bytes, key: bytes) -> bytes:
    return bytes(bytearray([key[i%len(key)] ^ d for i, d in enumerate(data)]))

def guess_keysize(ciphertext: bytes, min_size = 2, max_size = 40):
    result = []

    for ks in range(min_size, max_size):
        chunks = [
            ciphertext[i:i+ks] 
            for i in range(0,len(ciphertext),ks)
            ][:8]
        
        scores = []
        
        for i in range(len(chunks) - 1):
            for j in range(i+1, len(chunks)):
                if len(chunks[i]) != ks or len(chunks[j]) != ks:
                    continue

                scores.append(
                    hamming_distance(
                        chunks[i], chunks[j]
                        ) / ks
                )
        if not scores:
            continue
        avg_normalized = sum(scores)/len(scores)
        result.append((ks, avg_normalized))
    
    result.sort(key=lambda x: x[1])
    return [ks for ks, _ in result]

In [ ]:
def single_byte_xor(data: bytes, key: int):
    return bytes(bytearray([d ^ key for d in data]))

def score_english(text: bytes):
    score = 0

    freq = b"etaoinshrdlu ETAOINSHRDLU"

    for c in text:

        # printable + newline/tab
        if c in b"\n\r\t":
            score += 1

        elif c < 32 or c > 126:
            score -= 20
            continue

        if c in freq:
            score += 3

        if c == ord(" "):
            score += 5

        if chr(c).isalpha():
            score += 1

        if c in b"~@#$%^&*{}[]|":
            score -= 2

    return score

def crack_single_byte_xor(ciphertext: bytes):
    best = None

    for k in range(256):
        p = single_byte_xor(ciphertext,k)
        score = score_english(p)

        if best is None or score > best[3]:
            best = (
                ciphertext.hex(),
                k,
                p,
                score
            )

    return best

In [ ]:
def break_repeating_key_xor(ciphertext: bytes, keysize: int):
    blocks = [ciphertext[i:i+keysize] for i in range(0,len(ciphertext),keysize)]
    transpose = []

    for i in range(keysize):
        tmp = []
        for block in blocks:
            if i < len(block):
                tmp.append(block[i])
        transpose.append(bytes(tmp))

    key = b""
    for k in transpose:
        cand = crack_single_byte_xor(k)
        key += bytes([cand[1]])
    return key

In [ ]:
with open("/mnt/d/my-kisah/crypto/1.cryptopals/files/6.txt") as f:
    b64_data = "".join(
        line.strip() 
        for line in f
    )

ct = base64.b64decode(b64_data)
ks = guess_keysize(ct)

keys = []
for s in ks[:5]:
    print(
        s,
        break_repeating_key_xor(ct,s)
    )
    cand_key = break_repeating_key_xor(ct,s)

    print(f"Keysize: {s}")
    print(f"Key: {cand_key}")

    pt = repeating_key_xor(ct,cand_key)

    print(pt.decode(errors="ignore")[:300])
    print("="*50)

## Challenge 7
### AES in ECB mode

The Base64-encoded content in this file has been encrypted via AES-128 in ECB mode under the key

"YELLOW SUBMARINE".

(case-sensitive, without the quotes; exactly 16 characters; I like "YELLOW SUBMARINE" because it's exactly 16 bytes long, and now you do too).

Decrypt it. You know the key, after all.

Easiest way: use OpenSSL::Cipher and give it AES-128-ECB as the cipher.

### Do this with code.
You can obviously decrypt this using the OpenSSL command-line tool, but we're having you get ECB working in code for a reason. You'll need it a lot later on, and not just for attacking ECB.

In [ ]:
!pip install pycryptodome

In [ ]:
import base64
from Crypto.Cipher import AES

In [ ]:
def aes_ecb_decrypt(ciphertext: bytes, key: bytes):
    ct = AES.new(key, AES.MODE_ECB)
    pt = ct.decrypt(ciphertext)
    return pt

In [ ]:
with open("/mnt/d/my-kisah/crypto/1.cryptopals/files/7.txt") as f:
    b64_data = "".join(
        line.strip()
        for line in f
    )

cipher = base64.b64decode(b64_data)
key = b"YELLOW SUBMARINE"
plain = aes_ecb_decrypt(cipher, key)
print("".join(sentc for sentc in plain.decode()))

## Challenge 8
### Detect AES in ECB mode


In this file are a bunch of hex-encoded ciphertexts.

One of them has been encrypted with ECB.

Detect it.

Remember that the problem with ECB is that it is stateless and deterministic; the same 16 byte plaintext block will always produce the same 16 byte ciphertext.

In [209]:
def split_block(ciphertext: bytes, size = 16):
    blocks = []
    for i in range(0, len(ciphertext), size):
        blocks.append(ciphertext[i:i+size])
    return blocks

def check_repeated(block: list) -> int:
    block_length = len(block)
    uniq_block = len(set(block))
    diff = abs(block_length - uniq_block)
    if diff != 0:
        return diff
    return 0

In [210]:
with open("/mnt/d/my-kisah/crypto/1.cryptopals/files/8.txt") as f:
    h = f.readlines()

for idx, c in enumerate(h):
    ct = bytes.fromhex(c.strip())
    blocks = split_block(ct)
    repeated = check_repeated(blocks)
    if repeated > 0:
        print(f"Ciphertext      : {ct}")
        print(f"Hex Ciphertext  : {c.strip()}")
        print(f"Index ke        : {idx}")

Ciphertext      : b'\xd8\x80a\x97@\xa8\xa1\x9bx@\xa8\xa3\x1c\x81\n=\x08d\x9a\xf7\r\xc0oO\xd5\xd2\xd6\x9ctL\xd2\x83\xe2\xdd\x05/kd\x1d\xbf\x9d\x11\xb04\x85B\xbbW\x08d\x9a\xf7\r\xc0oO\xd5\xd2\xd6\x9ctL\xd2\x83\x94u\xc9\xdf\xdb\xc1\xd4e\x97\x94\x9d\x9c~\x82\xbfZ\x08d\x9a\xf7\r\xc0oO\xd5\xd2\xd6\x9ctL\xd2\x83\x97\xa9>\xab\x8dj\xec\xd5fH\x91Tx\x9ak\x03\x08d\x9a\xf7\r\xc0oO\xd5\xd2\xd6\x9ctL\xd2\x83\xd4\x03\x18\x0c\x98\xc8\xf6\xdb\x1f*?\x9c@@\xde\xb0\xabQ\xb2\x993\xf2\xc1#\xc5\x83\x86\xb0o\xba\x18j'
Hex Ciphertext  : d880619740a8a19b7840a8a31c810a3d08649af70dc06f4fd5d2d69c744cd283e2dd052f6b641dbf9d11b0348542bb5708649af70dc06f4fd5d2d69c744cd2839475c9dfdbc1d46597949d9c7e82bf5a08649af70dc06f4fd5d2d69c744cd28397a93eab8d6aecd566489154789a6b0308649af70dc06f4fd5d2d69c744cd283d403180c98c8f6db1f2a3f9c4040deb0ab51b29933f2c123c58386b06fba186a
Index ke        : 132
